1) Создать проект в Roboflow
Добавить классы apple, banana, orange.
2) Собрать датасет
Цель — 150 изображений. Если времени будет мало, начнём с меньшего количества.
3) Разметить фотографии
Bounding Boxes для каждого фрукта.
4) Обучить YOLO11
Используем Google Colab, чтобы не тратить время на настройку локального окружения.
5) Сохранить best.pt
Это веса нашей лучшей модели.
6) Создать Telegram-бота
Принимает фото, запускает детекцию и возвращает результат.

# Детекция объектов с помощью YOLO

In [30]:
# !pip install ultralytics
from ultralytics import YOLO

import torch

# !pip install roboflow
from roboflow import Roboflow

# pip install aiogram python-dotenv

In [31]:
device = 'cpu'

In [47]:
model = YOLO("yolo11n.pt")

In [37]:
results = model.predict("https://avatars.mds.yandex.net/i?id=a42095dcf3c911c779d124a36f363a64_l-9229153-images-thumbs&n=13", conf=0.25)


0: 384x640 1 boat, 91.4ms
Speed: 2.1ms preprocess, 91.4ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)


In [38]:
# Здесь мы обрабатываем результаты работы YOLO и выводим найденные объекты на изображения

# Process results list
for result in results:
    boxes = result.boxes  # bbox нужен для детекции
    masks = result.masks  # Маска нужна для сегментации
    keypoints = result.keypoints  # Для определения позы
    probs = result.probs  # Для классификации найденного объекта
    obb = result.obb  # Ориентированный bbox
    result.show()  # display to screen

# 2. Дообучение yolo11 (fine-tuning)

In [39]:
# !pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="vNkYNMmcieguRCAjb81B")
project = rf.workspace("s-workspace-apzdn").project("ships-iubeu")
version = project.version(1)
dataset = version.download("yolov11")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to ships-1 in yolov11:: 100%|██████████| 54/54 [00:00<00:00, 995.25it/s]


In [40]:
print(dataset.location)

d:\Elbrus bootcamp\2Phase\GitHub\ds-phase-2\09-cv\ships-1


In [41]:
model = YOLO("yolo11n.pt")

In [56]:
results = model.train(
    data="ships-1/data.yaml",
    epochs=30,
    imgsz=640, # Размер изображений при обучении — 640 × 640
    batch=16,
    device=device
    )

New https://pypi.org/project/ultralytics/8.4.218 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.216  Python-3.14.7 torch-2.15.0+cpu CPU (13th Gen Intel Core i5-13420H)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=ships-1/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=D:\Elbrus bootcamp\2Phase

3. Инференс(предсказание) дообученной модели

In [63]:
url = "https://i.pinimg.com/736x/84/98/47/8498476013be35c120c11d6cf91f34f3.jpg"

results = model.predict(url, conf=0.5)


Found https://i.pinimg.com/736x/84/98/47/8498476013be35c120c11d6cf91f34f3.jpg locally at 8498476013be35c120c11d6cf91f34f3.jpg
image 1/1 d:\Elbrus bootcamp\2Phase\GitHub\ds-phase-2\09-cv\8498476013be35c120c11d6cf91f34f3.jpg: 640x384 7 shipss, 45.3ms
Speed: 1.5ms preprocess, 45.3ms inference, 0.8ms postprocess per image at shape (1, 3, 640, 384)


In [65]:
# Process results list
for result in results:
    boxes = result.boxes  # bbox нужен для детекции
    masks = result.masks  # Маска нужна для сегментации
    keypoints = result.keypoints  # Для определения позы
    probs = result.probs  # Для классификации найденного объекта
    obb = result.obb  # Ориентированный bbox
    result.show()  # display to screen

In [64]:
for result in results:
    print(result.boxes)
    print(model.names)
    result.show()

ultralytics.engine.results.Boxes object with attributes:

cls: tensor([0., 0., 0., 0., 0., 0., 0.])
conf: tensor([0.9997, 0.9950, 0.9198, 0.7275, 0.6784, 0.6463, 0.6145])
data: tensor([[1.7942e+02, 9.4194e+01, 2.9197e+02, 5.3537e+02, 9.9974e-01, 0.0000e+00],
        [5.1863e+01, 9.1136e+01, 3.0162e+02, 5.3906e+02, 9.9498e-01, 0.0000e+00],
        [1.8543e+02, 8.6117e+01, 5.6240e+02, 5.3642e+02, 9.1978e-01, 0.0000e+00],
        [5.6793e+02, 9.1758e+02, 7.3600e+02, 1.3067e+03, 7.2752e-01, 0.0000e+00],
        [3.1594e+02, 2.8610e+02, 5.5768e+02, 5.1076e+02, 6.7838e-01, 0.0000e+00],
        [5.0170e+02, 3.5385e+02, 7.3600e+02, 9.1694e+02, 6.4627e-01, 0.0000e+00],
        [3.2343e+02, 3.4861e+02, 4.0545e+02, 4.7356e+02, 6.1447e-01, 0.0000e+00]])
id: None
is_track: False
orig_shape: (1308, 736)
shape: torch.Size([7, 6])
xywh: tensor([[ 235.6953,  314.7826,  112.5428,  441.1766],
        [ 176.7438,  315.0959,  249.7610,  447.9189],
        [ 373.9178,  311.2674,  376.9733,  450.3011],
     

In [66]:
# ищем лучшие веса
from pathlib import Path

files = list(Path(".").rglob("best.pt"))

for file in files:
    print(file)

runs\detect\train-9\weights\best.pt
runs\detect\train-8\weights\best.pt
runs\detect\train-7\weights\best.pt
runs\detect\train-6\weights\best.pt
runs\detect\train-3\weights\best.pt
runs\detect\train-11\weights\best.pt
runs\detect\train-10\weights\best.pt


In [67]:
# экспортируем модель
model.export

<bound method Model.export of YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, bias=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, bias=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C3k2(
        (cv1): Conv(
          (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, bias=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(48, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchN

# Объедиённыый датасет

In [74]:
# !pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="vNkYNMmcieguRCAjb81B")
project = rf.workspace("s-workspace-apzdn").project("ships-mountains-16lrf")
version = project.version(1)
dataset = version.download("yolov11")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to ships-mountains-1 in yolov11:: 100%|██████████| 212/212 [00:00<00:00, 2118.71it/s]


In [75]:
print(dataset.location)

d:\Elbrus bootcamp\2Phase\GitHub\ds-phase-2\09-cv\ships-mountains-1


In [76]:
results = model.train(
    data="ships-mountains-1",
    epochs=20,
    imgsz=640,
    batch=16,
    device=device
    )

New https://pypi.org/project/ultralytics/8.4.218 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.216  Python-3.14.7 torch-2.15.0+cpu CPU (13th Gen Intel Core i5-13420H)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=ships-mountains-1, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=D:\Elbrus bootcamp\2Phase

In [77]:
url = "https://avatars.mds.yandex.net/get-shedevrum/14766198/img_1c9f7f6f64bd11f085e056f044dcbc0f/orig"

results = model.predict(url, conf=0.5)


0: 640x640 1 mountains, 1 ships, 104.8ms
Speed: 7.8ms preprocess, 104.8ms inference, 0.7ms postprocess per image at shape (1, 3, 640, 640)


In [78]:
for result in results:
    print(result.boxes)
    print(model.names)
    result.show()

ultralytics.engine.results.Boxes object with attributes:

cls: tensor([1., 0.])
conf: tensor([0.8892, 0.8370])
data: tensor([[390.6060, 672.7349, 803.3322, 806.8671,   0.8892,   1.0000],
        [104.4598, 200.1177, 564.6216, 417.0796,   0.8370,   0.0000]])
id: None
is_track: False
orig_shape: (1024, 1024)
shape: torch.Size([2, 6])
xywh: tensor([[596.9691, 739.8010, 412.7263, 134.1322],
        [334.5407, 308.5987, 460.1618, 216.9619]])
xywhn: tensor([[0.5830, 0.7225, 0.4031, 0.1310],
        [0.3267, 0.3014, 0.4494, 0.2119]])
xyxy: tensor([[390.6060, 672.7349, 803.3322, 806.8671],
        [104.4598, 200.1177, 564.6216, 417.0796]])
xyxyn: tensor([[0.3815, 0.6570, 0.7845, 0.7880],
        [0.1020, 0.1954, 0.5514, 0.4073]])
{0: 'mountains', 1: 'ships'}


In [79]:
# ищем лучшие веса
from pathlib import Path

files = list(Path(".").rglob("best.pt"))

for file in files:
    print(file)

runs\detect\train-9\weights\best.pt
runs\detect\train-8\weights\best.pt
runs\detect\train-7\weights\best.pt
runs\detect\train-6\weights\best.pt
runs\detect\train-3\weights\best.pt
runs\detect\train-12\weights\best.pt
runs\detect\train-11\weights\best.pt
runs\detect\train-10\weights\best.pt


In [80]:
# экспортируем модель
model.export

<bound method Model.export of YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, bias=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, bias=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C3k2(
        (cv1): Conv(
          (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, bias=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(48, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchN

In [81]:
from ultralytics import YOLO

for name in ["train-10", "train-11", "train-12"]:
    model = YOLO(f"runs/detect/{name}/weights/best.pt")
    print(name, model.names)

train-10 {0: 'ships'}
train-11 {0: 'ships'}
train-12 {0: 'mountains', 1: 'ships'}


In [ ]:
# pip install aiogram python-dotenv

import asyncio
import os
from pathlib import Path
from io import BytesIO

import streamlit
from ultralytics import YOLO
from PIL import Image

from aiogram import Bot, Dispatcher, F
from aiogram.filters import CommandStart
from aiogram.types import Message, BufferedInputFile

from dotenv import load_dotenv


# Загружаем токен
load_dotenv()

BOT_TOKEN = os.getenv("BOT_TOKEN")

# Загружаем последнюю модель
from pathlib import Path

MODEL_PATH = Path("runs/detect/train-12/weights/best.pt")

model = YOLO(str(MODEL_PATH))

# Создаём бота и диспетчер
bot = Bot(token=BOT_TOKEN)
dp = Dispatcher()


# Команда /start
@dp.message(CommandStart())
async def start(message: Message):
    await message.answer(
        "Привет! 🚢⛰️\n"
        "Отправь мне фотографию, и я найду на ней корабли и горы."
    )


# Обработка фотографии
@dp.message(F.photo)
async def handle_photo(message: Message):

    await message.answer("Обрабатываю фотографию... 🔍")

    # Получаем фотографию
    photo = message.photo[-1]

    file_info = await bot.get_file(photo.file_id)
    downloaded_file = await bot.download_file(file_info.file_path)

    # Открываем изображение
    image = Image.open(downloaded_file).convert("RGB")

    # Предсказание модели
    results = model.predict(image, conf=0.25)

    # Рисуем рамки без подписей классов
    result_image = results[0].plot(labels=False)

    # Переводим BGR в RGB
    result_image = result_image[:, :, ::-1]

    # Сохраняем изображение в память
    output = BytesIO()
    Image.fromarray(result_image).save(output, format="JPEG")

    # Отправляем результат
    result_file = BufferedInputFile(
        output.getvalue(),
        filename="prediction.jpg"
    )

    await message.answer_photo(
        photo=result_file
    )


# Если отправили не фотографию
@dp.message()
async def other_messages(message: Message):
    await message.answer("Отправь мне фотографию для распознавания 😊")


# Запуск бота
async def main():
    await dp.start_polling(bot)


await main()

Failed to fetch updates - TelegramConflictError: Telegram server says - Conflict: terminated by other getUpdates request; make sure that only one bot instance is running
Sleep for 1.000000 seconds and try again... (tryings = 0, bot id = 9040669380)
Failed to fetch updates - TelegramConflictError: Telegram server says - Conflict: terminated by other getUpdates request; make sure that only one bot instance is running
Sleep for 1.208094 seconds and try again... (tryings = 1, bot id = 9040669380)
Failed to fetch updates - TelegramConflictError: Telegram server says - Conflict: terminated by other getUpdates request; make sure that only one bot instance is running
Sleep for 1.672819 seconds and try again... (tryings = 2, bot id = 9040669380)
Failed to fetch updates - TelegramConflictError: Telegram server says - Conflict: terminated by other getUpdates request; make sure that only one bot instance is running
Sleep for 2.090088 seconds and try again... (tryings = 3, bot id = 9040669380)
Fail

CancelledError: 

Failed to fetch updates - TelegramConflictError: Telegram server says - Conflict: terminated by other getUpdates request; make sure that only one bot instance is running
Sleep for 5.131783 seconds and try again... (tryings = 22, bot id = 9040669380)
Failed to fetch updates - TelegramConflictError: Telegram server says - Conflict: terminated by other getUpdates request; make sure that only one bot instance is running
Sleep for 4.969867 seconds and try again... (tryings = 23, bot id = 9040669380)
Failed to fetch updates - TelegramConflictError: Telegram server says - Conflict: terminated by other getUpdates request; make sure that only one bot instance is running
Sleep for 5.288904 seconds and try again... (tryings = 24, bot id = 9040669380)
Failed to fetch updates - TelegramConflictError: Telegram server says - Conflict: terminated by other getUpdates request; make sure that only one bot instance is running
Sleep for 4.902825 seconds and try again... (tryings = 25, bot id = 9040669380)


@BotFather - \newboot - имя бота

Как подключить его к нашему проекту
Создадим файл .env:
BOT_TOKEN=твой_токен

А в Python получим его:

from dotenv import load_dotenv
import os

load_dotenv()

BOT_TOKEN = os.getenv("BOT_TOKEN")

И передадим боту:

In [ ]:
# bot = Bot(token=BOT_TOKEN)

In [ ]:
# /mybots - ship and mountaince

In [ ]:
# /start